<a href="https://colab.research.google.com/github/sarita380/5BTelemedicineAppSol/blob/master/Homework6_PCA_Spark_Scala.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless
!java -version


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  openjdk-17-jre-headless
Suggested packages:
  openjdk-17-demo openjdk-17-source libnss-mdns fonts-dejavu-extra
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  openjdk-17-jdk-headless openjdk-17-jre-headless
0 upgraded, 2 newly installed, 0 to remove and 1 not upgraded.
Need to get 120 MB of archives.
After this operation, 272 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 openjdk-17-jre-headless amd64 17.0.20.1+1-1~24.04 [48.1 MB]
Get:2 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 openj

In [2]:

!wget -q https://archive.apache.org/dist/spark/spark-3.5.7/spark-3.5.7-bin-hadoop3.tgz
!tar -xzf spark-3.5.7-bin-hadoop3.tgz
!mv spark-3.5.7-bin-hadoop3 /content/spark
!/content/spark/bin/spark-shell --version


[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 3.5.7
      /_/
                        
Using Scala version 2.12.18, OpenJDK 64-Bit Server VM, 21.0.12.1
Branch HEAD
Compiled by user runner on 2025-09-17T20:37:30Z
Revision ed00d046951a7ecda6429accd3b9c5b2d

In [3]:

import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark"

os.environ["PATH"] = (
    os.environ["JAVA_HOME"] + "/bin:" +
    os.environ["SPARK_HOME"] + "/bin:" +
    os.environ["PATH"]
)

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("SPARK_HOME:", os.environ["SPARK_HOME"])
print("Environment configured!")


JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64
SPARK_HOME: /content/spark
Environment configured!


Upload exPCA.mat and jmatio.jar

In [6]:

from google.colab import files

uploaded = files.upload()



Saving jmatio.jar to jmatio (2).jar
Saving exPCA.mat to exPCA.mat


In [7]:

import os

print("MATLAB dataset:",
      os.path.exists("/content/exPCA.mat"))

print("JMatIO library:",
      os.path.exists("/content/jmatio.jar"))


MATLAB dataset: True
JMatIO library: True


Matlab using Scale

In [8]:

scala_code = """
import com.jmatio.io.MatFileReader
import com.jmatio.types.MLDouble

val reader = new MatFileReader("/content/exPCA.mat")
val content = reader.getContent()

val data = content.get("X")
  .asInstanceOf[MLDouble].getArray()

println("Number of rows: " + data.length)
println("Number of columns: " + data(0).length)

println("First 5 observations:")
data.take(5).foreach(row => println(row.mkString(", ")))

System.exit(0)
"""

with open("/content/check_data.scala", "w") as f:
    f.write(scala_code)

import subprocess

result = subprocess.run(
    ["/content/spark/bin/spark-shell",
     "--jars", "/content/jmatio.jar",
     "-i", "/content/check_data.scala"],
    input=":quit\n",
    text=True,
    capture_output=True,
    timeout=180
)

print(result.stdout[-5000:])
print(result.stderr[-1500:])
print("Exit code:", result.returncode)


Spark context Web UI available at http://49fffdef7fb9:4040
Spark context available as 'sc' (master = local[*], app id = local-1793159127147).
Spark session available as 'spark'.
java.lang.NoClassDefFoundError: sun/misc/Cleaner
  at com.jmatio.io.MatFileReader$1.run(Unknown Source)
  at java.base/java.security.AccessController.doPrivileged(AccessController.java:318)
  at com.jmatio.io.MatFileReader.clean(Unknown Source)
  at com.jmatio.io.MatFileReader.read(Unknown Source)
  at com.jmatio.io.MatFileReader.<init>(Unknown Source)
  at com.jmatio.io.MatFileReader.<init>(Unknown Source)
  ... 60 elided
Caused by: java.lang.ClassNotFoundException: sun.misc.Cleaner
  at java.base/java.net.URLClassLoader.findClass(URLClassLoader.java:445)
  at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:592)
  at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:525)
  ... 66 more
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ ._

In [9]:

import os
import subprocess

# Download a newer JMatIO version from Maven Central
url = (
    "https://repo.maven.apache.org/maven2/"
    "net/sourceforge/jmatio/jmatio/1.0/"
    "jmatio-1.0.jar"
)

result = subprocess.run(
    ["wget", "-O", "/content/jmatio-new.jar", url],
    capture_output=True,
    text=True
)

print("Download exit code:", result.returncode)

if result.returncode == 0:
    print("Library size:",
          os.path.getsize("/content/jmatio-new.jar"))
else:
    print(result.stderr[-2000:])



Download exit code: 0
Library size: 66536


In [10]:

import subprocess

result = subprocess.run(
    [
        "/content/spark/bin/spark-shell",
        "--jars", "/content/jmatio-new.jar",
        "-i", "/content/check_data.scala"
    ],
    input=":quit\n",
    text=True,
    capture_output=True,
    timeout=180
)

print(result.stdout[-5000:])
print(result.stderr[-1500:])
print("Exit code:", result.returncode)


Spark context Web UI available at http://49fffdef7fb9:4040
Spark context available as 'sc' (master = local[*], app id = local-1793159399947).
Spark session available as 'spark'.
java.lang.NoClassDefFoundError: sun/misc/Cleaner
  at com.jmatio.io.MatFileReader$1.run(MatFileReader.java:375)
  at java.base/java.security.AccessController.doPrivileged(AccessController.java:318)
  at com.jmatio.io.MatFileReader.clean(MatFileReader.java:366)
  at com.jmatio.io.MatFileReader.read(MatFileReader.java:321)
  at com.jmatio.io.MatFileReader.<init>(MatFileReader.java:154)
  at com.jmatio.io.MatFileReader.<init>(MatFileReader.java:102)
  ... 60 elided
Caused by: java.lang.ClassNotFoundException: sun.misc.Cleaner
  at java.base/java.net.URLClassLoader.findClass(URLClassLoader.java:445)
  at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:592)
  at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:525)
  ... 66 more
Welcome to
      ____              __
     / __/__  ___ _____/ /__


In [11]:

import subprocess

result = subprocess.run(
    [
        "/content/spark/bin/spark-shell",
        "--jars", "/content/jmatio-new.jar",
        "-i", "/content/check_data.scala"
    ],
    input=":quit\n",
    text=True,
    capture_output=True,
    timeout=180
)

print("=== SCALA OUTPUT ===")
print(result.stdout[-6000:])

print("=== ERRORS ===")
print(result.stderr[-2000:])

print("Exit code:", result.returncode)


=== SCALA OUTPUT ===
Spark context Web UI available at http://49fffdef7fb9:4040
Spark context available as 'sc' (master = local[*], app id = local-1793159519243).
Spark session available as 'spark'.
java.lang.NoClassDefFoundError: sun/misc/Cleaner
  at com.jmatio.io.MatFileReader$1.run(MatFileReader.java:375)
  at java.base/java.security.AccessController.doPrivileged(AccessController.java:318)
  at com.jmatio.io.MatFileReader.clean(MatFileReader.java:366)
  at com.jmatio.io.MatFileReader.read(MatFileReader.java:321)
  at com.jmatio.io.MatFileReader.<init>(MatFileReader.java:154)
  at com.jmatio.io.MatFileReader.<init>(MatFileReader.java:102)
  ... 60 elided
Caused by: java.lang.ClassNotFoundException: sun.misc.Cleaner
  at java.base/java.net.URLClassLoader.findClass(URLClassLoader.java:445)
  at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:592)
  at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:525)
  ... 66 more
Welcome to
      ____              __
     / _

Convert MATLAB to CSV

In [12]:

import scipy.io
import numpy as np
import pandas as pd

# Load MATLAB file
mat = scipy.io.loadmat("/content/exPCA.mat")

# Display variables
print("MATLAB variables:", mat.keys())

# Extract dataset
X = np.asarray(mat["X"], dtype=float)

print("Dataset shape:", X.shape)

# Convert to CSV without headers
pd.DataFrame(X).to_csv(
    "/content/exPCA.csv",
    index=False,
    header=False
)

print("CSV file created successfully!")
print("First 5 observations:")
print(X[:5])


MATLAB variables: dict_keys(['__header__', '__version__', '__globals__', 'X'])
Dataset shape: (50, 2)
CSV file created successfully!
First 5 observations:
[[3.38156267 3.38911268]
 [4.52787538 5.8541781 ]
 [2.65568187 4.41199472]
 [2.76523467 3.71541365]
 [2.84656011 4.17550645]]


Verify the CSV

In [13]:

import pandas as pd

df = pd.read_csv(
    "/content/exPCA.csv",
    header=None,
    names=["Feature_1", "Feature_2"]
)

print("Number of observations:", len(df))
print("Number of features:", len(df.columns))

display(df.head())


Number of observations: 50
Number of features: 2


,Feature_1,Feature_2
0,3.381563,3.389113
1,4.527875,5.854178
2,2.655682,4.411995
3,2.765235,3.715414
4,2.846560,4.175506


**Normalize the data using Scala**
Create and run the Scala normalization program

In [14]:

import subprocess

scala_code = """
import org.apache.spark.mllib.linalg.Vectors

// Load CSV data
val rawData = sc.textFile("/content/exPCA.csv")
  .map(line => line.split(",").map(_.toDouble))
  .cache()

// Calculate feature means
val n = rawData.count()

val mean1 = rawData.map(_(0)).sum() / n
val mean2 = rawData.map(_(1)).sum() / n

println("===== FEATURE MEANS =====")
println(s"Feature 1 Mean: $mean1")
println(s"Feature 2 Mean: $mean2")

// Normalize by subtracting means
val normalized = rawData.map { row =>
  Vectors.dense(
    row(0) - mean1,
    row(1) - mean2
  )
}.cache()

println("===== FIRST 5 NORMALIZED ROWS =====")
normalized.take(5).foreach(println)

// Verify normalized means
val check1 = normalized.map(_(0)).sum() / n
val check2 = normalized.map(_(1)).sum() / n

println("===== NORMALIZED MEANS =====")
println(s"Feature 1: $check1")
println(s"Feature 2: $check2")
"""

with open("/content/normalize.scala", "w") as f:
    f.write(scala_code)

result = subprocess.run(
    ["/content/spark/bin/spark-shell",
     "-i", "/content/normalize.scala"],
    input=":quit\n",
    text=True,
    capture_output=True,
    timeout=180
)

print(result.stdout[-7000:])
print(result.stderr[-1000:])


Spark context Web UI available at http://49fffdef7fb9:4040
Spark context available as 'sc' (master = local[*], app id = local-1793159928650).
Spark session available as 'spark'.
===== FEATURE MEANS =====
Feature 1 Mean: 3.9892652815843146
Feature 2 Mean: 5.002805850257663
===== FIRST 5 NORMALIZED ROWS =====
[-0.6077026149487548,-1.613693165365592]
[0.5386100988188929,0.8513722509117505]
[-1.3335834128448516,-0.5908111327728776]
[-1.2240306073592393,-1.287392200514371]
[-1.1427051753632202,-0.8272994007432777]
===== NORMALIZED MEANS =====
Feature 1: -1.0480505352461477E-15
Feature 2: 2.575717417130363E-16
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 3.5.7
      /_/
         
Using Scala version 2.12.18 (OpenJDK 64-Bit Server VM, Java 17.0.20.1)
Type in expressions to have them evaluated.
Type :help for more information.

scala> :quit

Setting default log level to "WARN".
To adjust logging level use 

**Calculate the top principal component using Spark PCA**

In [15]:

import subprocess

scala_code = """
import org.apache.spark.mllib.linalg.Vectors
import org.apache.spark.mllib.linalg.distributed.RowMatrix

// Load the dataset
val rawData = sc.textFile("/content/exPCA.csv")
  .map(line => line.split(",").map(_.toDouble))
  .cache()

// Calculate means
val n = rawData.count()
val mean1 = rawData.map(_(0)).sum() / n
val mean2 = rawData.map(_(1)).sum() / n

// Normalize the data
val normalized = rawData.map { row =>
  Vectors.dense(
    row(0) - mean1,
    row(1) - mean2
  )
}

// Create Spark RowMatrix
val matrix = new RowMatrix(normalized)

// Compute top principal component
val pc = matrix.computePrincipalComponents(1)

println()
println("================================")
println("   TOP PRINCIPAL COMPONENT")
println("================================")
println(pc)
println("================================")
"""

with open("/content/pca.scala", "w") as f:
    f.write(scala_code)

result = subprocess.run(
    [
        "/content/spark/bin/spark-shell",
        "-i", "/content/pca.scala"
    ],
    input=":quit\n",
    text=True,
    capture_output=True,
    timeout=180
)

print(result.stdout[-6000:])
print(result.stderr[-1500:])


Spark context Web UI available at http://49fffdef7fb9:4040
Spark context available as 'sc' (master = local[*], app id = local-1793160064623).
Spark session available as 'spark'.

   TOP PRINCIPAL COMPONENT
-0.76908153413682    
-0.6391506816469457  
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 3.5.7
      /_/
         
Using Scala version 2.12.18 (OpenJDK 64-Bit Server VM, Java 17.0.20.1)
Type in expressions to have them evaluated.
Type :help for more information.

scala> :quit

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/28 04:01:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable

[Stage 0:=============================>                             (1 + 1) / 2]

                                                                       

**Project all 50 observations onto PC1** | Calculate the 50 projected values

In [16]:

import subprocess

scala_code = """
import org.apache.spark.mllib.linalg.Vectors
import org.apache.spark.mllib.linalg.distributed.RowMatrix

// Load CSV data
val rawData = sc.textFile("/content/exPCA.csv")
  .map(line => line.split(",").map(_.toDouble))
  .cache()

// Calculate feature means
val n = rawData.count()
val mean1 = rawData.map(_(0)).sum() / n
val mean2 = rawData.map(_(1)).sum() / n

// Normalize data
val normalized = rawData.map { row =>
  Vectors.dense(
    row(0) - mean1,
    row(1) - mean2
  )
}

// Create RowMatrix
val matrix = new RowMatrix(normalized)

// Calculate top principal component
val pc = matrix.computePrincipalComponents(1)

// Project data onto PC1
val projected = matrix.multiply(pc)

// Collect all 50 projected values
val values = projected.rows.collect()

println()
println("================================")
println(" PROJECTED DATA - 50 VALUES")
println("================================")

values.zipWithIndex.foreach {
  case (v, i) =>
    println(f"${i + 1}%2d: ${v(0)}%.8f")
}

println("================================")
println("Total projected values: " + values.length)
"""

with open("/content/projection.scala", "w") as f:
    f.write(scala_code)

result = subprocess.run(
    [
        "/content/spark/bin/spark-shell",
        "-i", "/content/projection.scala"
    ],
    input=":quit\n",
    text=True,
    capture_output=True,
    timeout=180
)

print(result.stdout[-10000:])
print(result.stderr[-1000:])


Spark context Web UI available at http://49fffdef7fb9:4040
Spark context available as 'sc' (master = local[*], app id = local-1793160410036).
Spark session available as 'spark'.

 PROJECTED DATA - 50 VALUES
 1: 1.49876595
 2: -0.95839024
 3: 1.40325172
 4: 1.76421694
 5: 1.40760243
 6: -0.87367998
 7: 1.27050164
 8: -2.55067120
 9: -0.01469839
10: -0.83694188
11: -0.70212917
12: -0.58711016
13: -0.12493311
14: -0.74690506
15: 1.67629396
16: -2.10275704
17: -0.95949530
18: 1.11633715
19: -2.37070273
20: -0.69001651
21: -2.39397485
22: 0.44284714
23: -1.98340505
24: -0.01058959
25: 1.83205377
26: 0.62719172
27: -1.33171608
28: -1.45467270
29: 1.01919098
30: 0.01489202
31: -0.07212622
32: -1.78539513
33: 1.41318051
34: -0.82644523
35: 0.75167377
36: -1.40551081
37: 1.82309802
38: -1.59458841
39: 2.80783613
40: -0.32551527
41: -0.98578762
42: 0.98465469
43: 1.38952836
44: -1.03742062
45: 1.87686597
46: -0.24535117
47: 3.51800218
48: 1.54860441
49: 0.34412682
50: -1.55978675
Total projected